# 35 — Plan-and-Execute Agent

Separate planning from execution: a planner commits to a full step list upfront, an executor runs each step in order, and a synthesizer combines all outputs.

**What you'll learn**
- Why upfront planning reduces token spend vs interleaved ReAct loops
- `with_structured_output(Plan)` — force the LLM to emit a typed Pydantic object
- Executor loop pattern: consume one step per iteration, carry results forward
- `Annotated[list[str], operator.add]` — append-only reducer for accumulated results
- Synthesizer: combine N step outputs into one coherent final answer

**Contrast:** `4-basic-react-agent` — ReAct interleaves observe→think→act; Plan-Execute commits to the full plan before touching any tool.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langgraph python-dotenv pydantic

In [ ]:
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()
if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Plan-Execute vs ReAct ──────────────────────────────────────────────────
#
# ReAct loop (example 4):         Plan-Execute (this example):
#   think → act → observe           plan ALL steps upfront
#   think → act → observe             execute step 1
#   think → act → observe             execute step 2
#   ...                               execute step N
#   answer                          synthesize → answer
#
# ReAct: adaptive, can replan on observation.
# Plan-Execute: cheaper (one plan call + N executor calls), more predictable.
#
# Key primitive: with_structured_output(Plan)
#   Forces the LLM to return a validated Pydantic object instead of free text.
#   No parsing, no regex — the output is already typed.

print("Plan-Execute: commit to full plan, execute each step, synthesize.")
print("ReAct:        interleave observe-think-act, can adapt mid-task.")

In [ ]:
# ── 4. Build the plan-execute graph ──────────────────────────────────────────
import operator
from typing import Annotated, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from pydantic import BaseModel


class Step(BaseModel):
    step: str


class Plan(BaseModel):
    steps: list[Step]


class PlanState(TypedDict):
    task: str
    steps: list[str]  # remaining steps to execute
    results: Annotated[list[str], operator.add]  # append-only: each executor call adds one
    final_answer: str


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
planner_llm = llm.with_structured_output(Plan)

PLANNER_SYSTEM = (
    "You are a task planner. Break the user's task into 3-5 clear, sequential steps. "
    "Each step should be a concrete action. Return ONLY the structured plan."
)
EXECUTOR_SYSTEM = (
    "You are a task executor. Complete the given step in 2-3 sentences. "
    "Reference prior results where relevant."
)
SYNTHESIZER_SYSTEM = (
    "Combine all completed step results into a coherent final response. "
    "Be concise and well-structured."
)


def planner(state: PlanState) -> dict:
    plan = planner_llm.invoke([SystemMessage(PLANNER_SYSTEM), HumanMessage(state["task"])])
    steps = [s.step for s in plan.steps]
    print(f"  Plan ({len(steps)} steps): {steps}")
    return {"steps": steps, "results": []}


def executor(state: PlanState) -> dict:
    step = state["steps"][0]
    prior = "\n".join(f"  - {r}" for r in state["results"])
    prompt = f"Task: {state['task']}\n\nPrior results:\n{prior}\n\nExecute: {step}"
    result = llm.invoke([SystemMessage(EXECUTOR_SYSTEM), HumanMessage(prompt)])
    print(f"  ✓ {step[:55]}")
    return {"steps": state["steps"][1:], "results": [result.content]}


def synthesizer(state: PlanState) -> dict:
    combined = "\n".join(f"Step {i + 1}: {r}" for i, r in enumerate(state["results"]))
    prompt = f"Task: {state['task']}\n\nCompleted steps:\n{combined}"
    answer = llm.invoke([SystemMessage(SYNTHESIZER_SYSTEM), HumanMessage(prompt)])
    return {"final_answer": answer.content}


def should_continue(state: PlanState) -> str:
    return "executor" if state["steps"] else "synthesizer"


graph = StateGraph(PlanState)
graph.add_node("planner", planner)
graph.add_node("executor", executor)
graph.add_node("synthesizer", synthesizer)
graph.set_entry_point("planner")
graph.add_edge("planner", "executor")
graph.add_conditional_edges(
    "executor", should_continue, {"executor": "executor", "synthesizer": "synthesizer"}
)
graph.add_edge("synthesizer", END)
app = graph.compile()
print("Graph compiled.")

In [ ]:
# ── 5. Run the plan-execute agent ─────────────────────────────────────────────
TASK = "Write a 3-step launch plan for a developer tool startup."

print(f"TASK: {TASK}\n")
result = app.invoke({"task": TASK, "steps": [], "results": [], "final_answer": ""})

print(f"\n{'=' * 60}")
print(f"STEPS EXECUTED: {len(result['results'])}")
print(f"\nFINAL ANSWER:\n{result['final_answer']}")

## Exercises

**Exercise 1 — Inspect the structured output:** Print `plan` right after `planner_llm.invoke(...)`. What type is it? How does Pydantic validation catch a malformed step?

**Exercise 2 — Add replanning:** After synthesis, if `final_answer` contains "unclear", route back to planner with updated context. This creates a Plan-Execute-Replan loop.

**Exercise 3 — Count LLM calls:** Add a counter to each node. Compare total calls to `4-basic-react-agent` on the same task.

**Exercise 4 — Parallel steps:** Use `asyncio.gather` inside executor to run all steps concurrently. When is this safe (independent steps) vs unsafe (steps depend on each other)?

## What's next

- **37-rewoo-agent** — emit ALL tool calls upfront as a plan, execute in bulk, synthesize
- **4-basic-react-agent** — interleaved ReAct for adaptive open-ended tasks
- **18-self-reflecting-agent** — add critique/revise loop after synthesis